# Four-Dimensional Ensemble–Variational Data Assimilation (4DEnVar)
In previous tutorial, we explored 3DEnVar.
3DEnVar introduces ensemble-derived covariances into a variational framework, but still operates at a single analysis time, which ignores the temporal evolution of the error covariances within the data assimilation window. The 4DEnVar method represents a natural extension of 3DEnVar which aims to address this issue.


## 1. Introduction - From 3DEnVar to 4DEnVar
While 3DEnVar incorporates flow-dependent covariances from an ensemble, it does so at a single time. Observations are effectively treated as if they occur simultaneously, even when they are distributed across a time window. In contrast, 4DVar handles time-evolving observations by propagating information forward and backward using the tangent linear and adjoint models - but at the cost of significant implementation complexity. 4DEnVar addresses the issue of the time evolving observations without the use of tangent linear and adjoint of the numerical model. 

The key idea of 4DEnVar is as follows: Instead of evolving sensitivities using an adjoint model (as in 4DVar), **4DEnVar uses an ensemble of time-evolving forecasts to represent how uncertainties propagate in time.** For mathematical formula, please refer to Wang and Lei (2014) and Lorenc et al. (2015). Specifically, in 4DEnVar, we work with **an ensemble of model trajectories over a time window**. These trajectories - forecasts of each ensemble member throughout the DA time window - provide a time-dependent estimate of the system’s error statistics. Using these ensemble members, we can:
* Represent how perturbations at the initial time evolve forward in time,
* Relate the model state at the analysis time to observations at later times,

This allows observations at different times to influence the analysis through ensemble-derived temporal correlations, rather than through an adjoint model.


## 2. Practical implementation of 4DEnVar in JEDI

In the JEDI implementation of 4DEnVar, the assimilation window is divided into subwindows, and the background is specified as a list of states at the boundaries of these subwindows. For example, with a 24-hour assimilation window and a 6-hour subwindow as in our experiments, we specify background states at five times: $t_0$,  $t_0 + 6$,  $t_0 + 12$,  $t_0 + 18$,  and  $t_0 + 24$ hours.

The ensemble members need to be specified at each subwindow boundary. This ensures that the ensemble perturbations at each time are available directly from the input files.

The background state list and ensemble trajectory list are specified in parallel. For example, for a 24-hour window with 6-hour subwindows:

```yaml
background:
  states:
  - date: 2009-12-30T00:00:00Z
    filename: bg_T+0h.nc
  - date: 2009-12-30T06:00:00Z
    filename: bg_T+6h.nc
  - date: 2009-12-30T12:00:00Z
    filename: bg_T+12h.nc
  - date: 2009-12-30T18:00:00Z
    filename: bg_T+18h.nc
  - date: 2009-12-31T00:00:00Z
    filename: bg_T+24h.nc
```

And similarly for the ensemble:

```yaml
members from template:
  template:
    states:
    - date: 2009-12-30T00:00:00Z
      filename: ens_%mem%_T+0h.nc
    - date: 2009-12-30T06:00:00Z
      filename: ens_%mem%_T+6h.nc
    - date: 2009-12-30T12:00:00Z
      filename: ens_%mem%_T+12h.nc
    - date: 2009-12-30T18:00:00Z
      filename: ens_%mem%_T+18h.nc
    - date: 2009-12-31T00:00:00Z
      filename: ens_%mem%_T+24h.nc
```

This structure makes the time-dependent nature of the ensemble covariance explicit and ensures that JEDI has access to the full ensemble trajectory over the assimilation window.

## 3. Prerequisites

Before proceeding with the 4DEnVar experiments, we need to ensure that the necessary input files are available. These files were generated in previous tutorials:

1. **Truth and Background:** The truth provides the synthetic observations, and the background is the forecast to be corrected. These were generated in the 4DVar tutorial. The background must be available at multiple times (the beginning and end of the assimilation window, and at any intermediate subwindow boundaries).

2. **Ensemble Trajectories:** An ensemble of background forecasts over the assimilation window is required. Each ensemble member must be available at the same times as the background. These were generated in the **0.qg_tutorial_start** tutorial.

3. **Observations:** Synthetic observations extracted from the truth at specified locations and times. We will use an observation at $t_0 + 1h$ (early in the window), and compare with an observation at $t_0 + 24h$ (end of window) to study the impact of observation timing. We will create these observations within this tutorial.

4. **Environment Variables:**
   - `$JEDI_BIN`: Directory containing the JEDI executables
   - `$JEDI_EDU`: Root directory of the JEDI education materials (maps to `/home/nonroot/shared/EDU` in Docker)

Let us verify that the environment is correctly configured:

In [ ]:
# Set up environment variables
export JEDI_BIN=/home/nonroot/build/bin
export JEDI_EDU=/home/nonroot/shared/EDU

# Verify environment
echo "JEDI_BIN: $JEDI_BIN"
echo "JEDI_EDU: $JEDI_EDU"
echo ""

# Check for required background files at multiple times
echo "Checking background trajectory files..."
ls -lh $JEDI_EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00:00:00Z.PT0S.nc
ls -lh $JEDI_EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00:00:00Z.PT6H.nc
ls -lh $JEDI_EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00:00:00Z.PT12H.nc
ls -lh $JEDI_EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00:00:00Z.PT18H.nc
ls -lh $JEDI_EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00:00:00Z.P1D.nc
echo ""

# Check for ensemble trajectory files (sample first member)
echo "Checking ensemble trajectory files (member 1)..."
ls -lh $JEDI_EDU/qgstart/output/bg/bkgd.ens.1.2009-12-30T00:00:00Z.PT0S.nc
ls -lh $JEDI_EDU/qgstart/output/bg/bkgd.ens.1.2009-12-30T00:00:00Z.PT6H.nc
ls -lh $JEDI_EDU/qgstart/output/bg/bkgd.ens.1.2009-12-30T00:00:00Z.PT12H.nc
ls -lh $JEDI_EDU/qgstart/output/bg/bkgd.ens.1.2009-12-30T00:00:00Z.PT18H.nc
ls -lh $JEDI_EDU/qgstart/output/bg/bkgd.ens.1.2009-12-30T00:00:00Z.P1D.nc


If any files are missing (i.e. you get an error above), please complete the`0.qg_tutorial_start` tutorial first. 

## 4. Experiments

We will conduct a series of experiments to explore the behavior of 4DEnVar. Each experiment modifies one aspect of the configuration:



### 4.1 Single Observation at the beginning of the window 

#### Create Observation

We will run the `qg_hofx.x` program to create the single observation, with the yaml file `makeobs4d_oneobs_begin` within the `qg4DEnVar/yamls` directory. The key parts are seen below. The longitude (`lon`) and latitude (`lat`) is set to 100 and 15, respectively. The `begin: PT1H` line sets that this observation is to be created one hour into the 24-h window.
```yaml
      generate:
        begin: PT1H           # Create observation 1 hour into window
        nval: 1
        obs locations:
          lon: [-125]          # List of longitudes for generated observations (only one here)
          lat: [15]           # List of latitudes for generated observations (only one here)
          z: [7000.0]         # List of heights for generated observations (only one here)
        obs error: 4.0e6      # Observation error standard deviation, in m^2/s
        obs period: PT24H
```


In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_hofx.x ./qg4DEnVar/yamls/makeobs4d_oneobs_begin.yaml

#### Run 4DEnVar

The configuration is specified in the YAML file `4dEnVar_oneobs_begin.yaml`. Navigate to the directory `$JEDI_EDU/qg4DEnVar/yamls` and examine this file. Below we show some key components to this file:

```yaml
cost function:
  cost type: 4D-Ens-Var               # Four-dimensional ensemble-variational method
  window begin: 2009-12-30T00:00:00Z
  window length: PT24H                # Total window for DA (24 h)
  subwindow: PT6H                     # Defines length of subwindows in the full DA window
  analysis variables: [x]              
  background:                         # Background trajectory: states at subwindow boundaries
    states:
    - date: 2009-12-30T00:00:00Z
      filename: qgstart/output/bg/bkgd.fc.2009-12-30T00:00:00Z.PT0S.nc
    - date: 2009-12-30T06:00:00Z
      filename: qgstart/output/bg/bkgd.fc.2009-12-30T00:00:00Z.PT6H.nc
    - date: 2009-12-30T12:00:00Z
      filename: qgstart/output/bg/bkgd.fc.2009-12-30T00:00:00Z.PT12H.nc
    - date: 2009-12-30T18:00:00Z
      filename: qgstart/output/bg/bkgd.fc.2009-12-30T00:00:00Z.PT18H.nc
    - date: 2009-12-31T00:00:00Z
      filename: qgstart/output/bg/bkgd.fc.2009-12-30T00:00:00Z.P1D.nc
  background error:
    covariance model: hybrid
    components:
    - covariance:
        covariance model: QgError      # Static-B component of covariances
        horizontal_length_scale: 2.2e6
        maximum_condition_number: 1.0e6
        standard_deviation: 1.8e7
        vertical_length_scale: 15000.0
      weight:
        value: 0.0                     # Weight for static covariance (set to 0 effectively turns off static-B)
    - covariance:
        covariance model: ensemble     # Ensemble component of covariances
        localization:
          horizontal_length_scale: 4.0e6 # 4000km localalization scale
          localization method: QG
          maximum_condition_number: 1.0e6
          standard_deviation: 1.0
          vertical_length_scale: 30000.0  # Essentially ignores vertical localization since scale >>> 10km
        members from template:          # Ensemble trajectories: states at subwindow boundaries for each member
          template:
            states:
            - date: 2009-12-30T00:00:00Z
              filename: qgstart/output/bg/bkgd.ens.%mem%.2009-12-30T00:00:00Z.PT0S.nc
            - date: 2009-12-30T06:00:00Z
              filename: qgstart/output/bg/bkgd.ens.%mem%.2009-12-30T00:00:00Z.PT6H.nc
            - date: 2009-12-30T12:00:00Z
              filename: qgstart/output/bg/bkgd.ens.%mem%.2009-12-30T00:00:00Z.PT12H.nc
            - date: 2009-12-30T18:00:00Z
              filename: qgstart/output/bg/bkgd.ens.%mem%.2009-12-30T00:00:00Z.PT18H.nc
            - date: 2009-12-31T00:00:00Z
              filename: qgstart/output/bg/bkgd.ens.%mem%.2009-12-30T00:00:00Z.P1D.nc 
          pattern: %mem%
          nmembers: 20
      weight:
        value: 1.0                      # Weight for ensemble  covariance (1.0 -> pure ensemble 4DEnVar)
    .
    . 
    .
```

#### Key Configuration Elements

**Window and Subwindows:** The 24-hour assimilation window is divided into four 6-hour subwindows. This means that **five** background and ensemble trajectories must be provided at $t_0$, $t_0 + 6$, $t_0 + 12$, $t_0 + 18$ and $t_0 + 24$ hours; hence why there are five files provided for background and ensembles in the yaml.

**Pure Ensemble Covariances:** The yaml file demonstrates how 4DEnVar may be run in hybrid mode using a blend of hybrid and static covariances; however, for these experiments we will set the static component weight to 0.0, effectively running a pure ensemble form of 4DEnVar.

**Localization:** The ensemble component uses a horizontal localization length scale of 4.0 × 10$^6$ m (4000 km).

**Observation:** A single streamfunction observation at $t_0 + 1h$ (early in the window) is assimilated.

#### Running the Experiment

We now run the 4DEnVar algorithm with this configuration:

> <u>**NOTE!!!**</u> The JEDI version of 4DEnVar **requires** Message Passing Interface (MPI) to complete. An MPI program runs multiple independent processes simultaneously, often with one process assigned per CPU core. Processes interact by sending and receiving messages to share data.
>
> In this case,  because we defined five output times for the subwindows, then we must have **at least 5 MPI processes** to run the 4DEnVar
>
> The command "mpirun" initializes a compiled MPI program, and the paramete "-n 5" specifies that we want to launch 5 processes to run this program.
>
> > *!!! Also note this may take 5-10 minutes to run, so be patient !!!*

In [ ]:
cd $JEDI_EDU
begin=`date`
mpirun -n 5 $JEDI_BIN/qg_4dvar.x ./qg4DEnVar/yamls/4dEnVar_oneobs_begin.yaml
echo STARTED AT $begin
echo ENDED AT `date`

#### Visualizing the Analysis Increment

After the minimization completes, we visualize the analysis increment (analysis minus background) to see how the single observation has influenced the analysis. We use the plotting script from the `plots_scripts` directory:

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qg4DEnVar/output/exp_begin/da/4DEnVar.an.2009-12-30T00\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00:00:00Z.PT0S.nc \
        --plotObsValues $JEDI_EDU/qg4DEnVar/output/exp_begin/obs/truth_oneob.begin.nc \
        --plotwind --plotstream --fieldmax 15000000.0 \
        --output $JEDI_EDU/qg4DEnVar/output/exp_begin/plots/4DEnVar_inc_00 --title "Increment 00H: obs at beginning of window"  
python ./plot.py qg fields \
        $JEDI_EDU/qg4DEnVar/output/exp_begin/da/4DEnVar.an.2009-12-30T12\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00:00:00Z.PT12H.nc \
        --plotObsValues $JEDI_EDU/qg4DEnVar/output/exp_begin/obs/truth_oneob.begin.nc \
        --plotwind --plotstream --fieldmax 15000000.0  \
        --output $JEDI_EDU/qg4DEnVar/output/exp_begin/plots/4DEnVar_inc_12 --title "Increment 12H: obs at beginning of window"  
python ./plot.py qg fields \
        $JEDI_EDU/qg4DEnVar/output/exp_begin/da/4DEnVar.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00:00:00Z.P1D.nc \
        --plotObsValues $JEDI_EDU/qg4DEnVar/output/exp_begin/obs/truth_oneob.begin.nc \
        --plotwind --plotstream --fieldmax 15000000.0  \
        --output $JEDI_EDU/qg4DEnVar/output/exp_begin/plots/4DEnVar_inc_24 --title "Increment 24H: obs at beginning of window"  
ls $JEDI_EDU/qg4DEnVar/output/exp_begin/plots
    #    display < $JEDI_EDU/qg4Dvar/output/exp_begin/plots/4dvar_inc_begin_x_diff.jpg

<center><img src="images/4DEnVar_inc_begin.gif" width="700"/></center>

**Questions for Reflection:**

Examine the increments at different times, try to answer the following:

- How do the increments evolve with time?
- Does it make sense? What dynamical process is being represented? (Hint: look at the streamfunction contours)

### 4.2 Single Observation at the end of the window

In this experiment, we investigate the behavior of 4DEnVar when the observation is instead available at the end of the DA window

#### Create Observation
We will run the `qg_hofx.x` program to create the single observation, with the yaml file `makeobs4d_oneobs_end.yaml` within the qg4DEnVar/yamls directory. 

First, examine the differences with the `makeobs4d_oneobs_begin.yaml` file, see if you understand them. Note that we have also moved this observation eastward, as we had done in the 4DVar tutorial as well.


In [ ]:
cd $JEDI_EDU
diff ./qg4DEnVar/yamls/makeobs4d_oneobs_begin.yaml ./qg4DEnVar/yamls/makeobs4d_oneobs_end.yaml

Now, create the observation and verify it was created

In [ ]:
cd $JEDI_EDU
$JEDI_BIN/qg_hofx.x ./qg4DEnVar/yamls/makeobs4d_oneobs_end.yaml

#### Experiment configuration

The yaml file is `4dEnVar_oneobs_end.yaml`. The only difference from the first experiment is the observation file.


In [ ]:
cd $JEDI_EDU
diff ./qg4DEnVar/yamls/4dEnVar_oneobs_begin.yaml ./qg4DEnVar/yamls/4dEnVar_oneobs_end.yaml 

In [ ]:
cd $JEDI_EDU
begin=`date`
mpirun -n 5 $JEDI_BIN/qg_4dvar.x ./qg4DEnVar/yamls/4dEnVar_oneobs_end.yaml
echo STARTED AT $begin
echo ENDED AT `date`

Now, run the 4DEnVar! Note as the first experiment, this will take 5-10 minutes to complete

#### Generating Plots

In [ ]:
cd $JEDI_EDU/plots_scripts
python ./plot.py qg fields \
        $JEDI_EDU/qg4DEnVar/output/exp_end/da/4DEnVar.an.2009-12-30T00\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00:00:00Z.PT0S.nc \
        --plotObsValues $JEDI_EDU/qg4DEnVar/output/exp_end/obs/truth_oneob.end.nc \
        --plotwind --plotstream --fieldmax 5000000.0 \
        --output $JEDI_EDU/qg4DEnVar/output/exp_end/plots/4DEnVar_inc_00 --title "Increment 00H: obs at end of window"  
python ./plot.py qg fields \
        $JEDI_EDU/qg4DEnVar/output/exp_end/da/4DEnVar.an.2009-12-30T12\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00:00:00Z.PT12H.nc \
        --plotObsValues $JEDI_EDU/qg4DEnVar/output/exp_end/obs/truth_oneob.end.nc \
        --plotwind --plotstream --fieldmax 5000000.0  \
        --output $JEDI_EDU/qg4DEnVar/output/exp_end/plots/4DEnVar_inc_12 --title "Increment 12H: obs at end of window"  
python ./plot.py qg fields \
        $JEDI_EDU/qg4DEnVar/output/exp_end/da/4DEnVar.an.2009-12-31T00\:00\:00Z.nc \
        $JEDI_EDU/qgstart/output/bg/bkgd.fc.2009-12-30T00:00:00Z.P1D.nc \
        --plotObsValues $JEDI_EDU/qg4DEnVar/output/exp_end/obs/truth_oneob.end.nc \
        --plotwind --plotstream --fieldmax 5000000.0  \
        --output $JEDI_EDU/qg4DEnVar/output/exp_end/plots/4DEnVar_inc_24 --title "Increment 24H: obs at end of window"  

<center><img src="images/4DEnVar_inc_end.gif" width="700"/></center>

#### Reflection
These experiments show how increments throughout the DA window are tied dynamically to each other through the model trajectories defined by the ensemble forecasts at subwindow border times

<br>

### The following is optional, but recommended to help complete the understanding of these experiments

We will plot what the ensemble variance errors look like throughout the window, as a supplement to the above increment plots. For instance, this may help further understand the question from the exp_begin experiment on why increments become more compact after 00H, and why increments seem to remain fixed on local contours on the eastward, leading edge of the local trough nearby the observation.

To plot these, we need to produce files at different times for each subwindow border. The following demonstrates how to use **bash for loops** to help more efficiently create the desired variance files and plots. The strategy is to create an array of the unique identifiers for each forecast time (thr forecast hours, e.g. 'PT6H', and the forecast dates, e.g. '30T06' which is the day and hour indentifier for each time). The bash program then uses the `sed` program to search and replace the template ens_variance.yaml with the %FHR% and %DAT% template strings, replacing them with the looped value for each index of the arrays.

In [ ]:
#2009-12-31T00:00:00Z
cd $JEDI_EDU
# Define arrays of forecast hours and dates (ddThh) for search and replace
fhrs=( PT0S  PT6H  PT12H PT18H P1D )
dats=( 30T00 30T06 30T12 30T18 31T00 )

# Get length of array for looping
array_length=${#fhrs[@]}

# Loop from index 0 to length-1, replace %DAT% and %FHR% in ense_variance.yaml with loop values, 
# and save to unique yaml files. Then run the ens_variance.x jedi program with each created yaml
# Note output is directed to log files in the yamls directory, if you would like to view them.
for (( i=0; i<array_length; i++ )); do
  echo "Index $i: ${fhrs[$i]} ${dats[$i]}"
  sed -e "s|%FHR%|${fhrs[$i]}|g;s|%DAT%|${dats[$i]}|g" ./qg4DEnVar/yamls/ens_variance.yaml \
                                                     > ./qg4DEnVar/yamls/ens_variance_${fhrs[$i]}.yaml

  $JEDI_BIN/qg_ens_variance.x ./qg4DEnVar/yamls/ens_variance_${fhrs[$i]}.yaml \
                           >& ./qg4DEnVar/yamls/ens_variance_${fhrs[$i]}.log
done




In [ ]:
# show an example difference of one of the produced ens_variance_${fhrs[$i]}.yaml 
# from the provided template ens_variance.yaml
cd $JEDI_EDU/qg4DEnVar/yamls
diff ens_variance.yaml ens_variance_P1D.yaml

# Verify that variance files were produced for all 5 subwindow border times
cd $JEDI_EDU
ls ./qg4DEnVar/output/exp_begin/da/variance*nc

The same looping strategy can be applied to plot the ensemble variances and background error files at different times. Note, be patient, this may take some time to run!

In [ ]:
# Plot ensemble variance for all times
cd $JEDI_EDU/plots_scripts
dats=( 30T00 30T06 30T12 30T18 31T00 ) # for ens variance plots
fhrs=( PT0S  PT6H  PT12H PT18H P1D ) # for bakground error plots
truthfhrs=( P15D P15DT6H P15DT12H P15DT18H P16D ) # for background error plots

# Get length of array for looping
array_length=${#fhrs[@]}

# Loop from index 0 to length-1, replace %DAT% and %FHR% in ense_variance.yaml with loop values, 
# and save to unique yaml files. Then run the ens_variance.x jedi program with each created yaml
# Note output is directed to log files in the yamls directory, if you would like to view them.
for (( i=0; i<array_length; i++ )); do
    python ./plot.py qg fields \
        $JEDI_EDU/qg4DEnVar/output/exp_begin/da/variance.diag.2009-12-${dats[$i]}\:00\:00Z.nc \
        --fieldmax 1e15 --plotObsValues $JEDI_EDU/qg4DEnVar/output/exp_begin/obs/truth_oneob.begin.nc \
        --output $JEDI_EDU/qg4DEnVar/output/exp_begin/plots/bkgd_ens_variance_${dats[$i]} --title "ens variance ${dats[$i]} "
done

<center><img src="images/ens_var.gif" width="700"/></center>


## 5. Summary

In this tutorial, we have explored Four-Dimensional Ensemble-Variational data assimilation (4DEnVar), which extends 3DEnVar into the time dimension. 

Key insights from this tutorial:

- **Flow-dependent error covariances** are essential for capturing the actual uncertainty in atmospheric forecasts. Static covariances provide a useful baseline but cannot adapt to the evolving synoptic situation.

- **Time-dependent ensemble trajectories** 4DEnVar naturally account for how forecast errors grow and propagate over the assimilation window. This is a significant advantage over 3DEnVar, which treats the ensemble as a snapshot at a single time.

- **Observation timing matters:** The 4D framework allows each observation to contribute information consistent with its timing within the window, improving the overall analysis quality compared to 3D methods that collapse all observations to a single time.